# Probe health with day-level embeddings
Following tutorial here: https://oxwearables.github.io/Sensori/tutorials/day-level-health/#prepare-the-analysis-data
The pre-processing notebook: tutorials/nhanes_preprocessing.ipynb(this outputs the .npy files same as my script is curently generating)
There is also the full tutorial notebook: tutorials/nhanes_evaluation.ipynb - I am re-working this to run the linear probe against e.g. sex, BMI etc to benchmark sensori's performance on lower back sensor data


In [1]:
from pathlib import Path

import torch

import os

from scripts.tutorial_nhanes.prepare_nhanes_tabular import prepare_tabular
from sensori.inference import extract_embeddings # looks dodgy but does work

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
torch.cuda.is_available()

/home/lucy/miniconda3/envs/Sensori/lib/python3.13/site-packages/torch/cuda/__init__.py:1074: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/lucy/miniconda3/envs/Sensori/lib/python3.13/site-packages/torch/cuda/__init__.py:1132: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


False

- first extract the embeddings - this code fetches the weights from huggingface

In [3]:
# Change this to the directory containing your preprocessed NHANES arrays.
PROCESSED_DATA_DIR = Path('/home/lucy/PycharmProjects/Sensori/data/sensori-data')
EMBEDDING_PATH = Path('mobilised_embeddings.npy')

# BATCH_SIZE = 100 needs about 70 GB of GPU memory for full-day inputs; reduce it if you run out.
BATCH_SIZE = 10
NUM_WORKERS = 6
SEED = 111

extract_embeddings(
    PROCESSED_DATA_DIR,
    output_path=EMBEDDING_PATH,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

Embeddings already exist: /home/lucy/PycharmProjects/Sensori/experiments/mobilised_embeddings.npy


PosixPath('/home/lucy/PycharmProjects/Sensori/experiments/mobilised_embeddings.npy')

## Prepare tabular data
So I think here I am prepping the demographic data to run the linear probe against

In [4]:

df = pd.read_csv(os.path.join("..", "data", "demographics.csv"))

df["BMI"] = (df["weight"]/(df["height"] * df["height"])) * 10000

# df

columns = ["Local Participant", "age", "gender", "BMI", "healthstat", "smoker", "alcocons"]

tabular_demo = df[columns]

tabular_demo

print(tabular_demo.columns)

Index(['Local Participant', 'age', 'gender', 'BMI', 'healthstat', 'smoker',
       'alcocons'],
      dtype='object')


In [5]:
# add in t25fw data
tfw_df = pd.read_csv(os.path.join("..", "data", "msfc.csv"))

cols = ["Local Participant","VISITNUM", "Trial", "T25FW1L", "TFWMN1N"]

tfw_df = tfw_df[cols]

tfw_df["T25FW"] = tfw_df["TFWMN1N"]

# Mapping
visit_map = {200: "t1", 210: "t2", 220: "t3", 230: "t4", 240: "t5"}
tfw_df["timestamp"] = tfw_df["VISITNUM"].map(visit_map)
tfw_df

,Local Participant,VISITNUM,Trial,T25FW1L,TFWMN1N,T25FW,timestamp
0,25146,200,1,T,10.6,10.6,t1
1,24566,200,1,T,5.9,5.9,t1
2,25239,200,1,T,5.9,5.9,t1
3,25238,200,1,T,5.5,5.5,t1
4,25237,230,4,T,16.6,16.6,t4
...,...,...,...,...,...,...,...
2404,10376,240,5,T,11.6,11.6,t5
2405,10376,230,4,T,12.9,12.9,t4
2406,10376,220,3,T,11.4,11.4,t3
2407,10376,210,2,T,8.3,8.3,t2


In [17]:
print(tfw_df.columns)

## merge tables

tabular = tfw_df.merge(
    tabular_demo,
    left_on="Local Participant",
    right_on="Local Participant",
    how="inner",
)

# encode binary categories
# Binary mapping
tabular['gender'] = tabular['gender'].map({'Male': 1, 'Female': 0})


# Boolean values directly convert to 1/0 with astype
tabular['smoker'] = tabular['smoker'].map({'T': 1, 'F': 0})

tabular

Index(['Local Participant', 'VISITNUM', 'Trial', 'T25FW1L', 'TFWMN1N', 'T25FW',
       'timestamp'],
      dtype='object')


,Local Participant,VISITNUM,Trial,T25FW1L,TFWMN1N,T25FW,timestamp,age,gender,BMI,healthstat,smoker,alcocons
0,25146,200,1,T,10.6,10.6,t1,65,0,27.343750,Less than average,0,1.0
1,24566,200,1,T,5.9,5.9,t1,39,0,22.246111,Average,0,0.0
2,25239,200,1,T,5.9,5.9,t1,64,0,23.423557,Less than average,1,0.0
3,25238,200,1,T,5.5,5.5,t1,52,0,24.609375,Average,0,0.0
4,25237,230,4,T,16.6,16.6,t4,64,1,34.602076,Good,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2404,10376,240,5,T,11.6,11.6,t5,50,1,22.790358,Good,1,2.0
2405,10376,230,4,T,12.9,12.9,t4,50,1,22.790358,Good,1,2.0
2406,10376,220,3,T,11.4,11.4,t3,50,1,22.790358,Good,1,2.0
2407,10376,210,2,T,8.3,8.3,t2,50,1,22.790358,Good,1,2.0


## build participant-level embeddings
- just gonna extract a few more locally to make this match the example better
- so what's going on here... L2 normalisation and then averaging of day embeddings???
- then join to tabular (outcome) data

In [18]:

def load_participant_embeddings(path):
    day_embeddings = np.load(path, allow_pickle=True).item()
    participants = {}
    for eid, days in day_embeddings.items():
        days = np.asarray(days, dtype=np.float32)
        norms = np.linalg.norm(days, axis=1)
        valid = np.isfinite(norms) & (norms > 0)
        if valid.any():
            participants[str(eid)] = (days[valid] / norms[valid, None]).mean(
                axis=0
            )

    frame = pd.DataFrame.from_dict(participants, orient="index")
    frame.index.name = "eid"  # Retains 12345_T1 as the index name
    frame.columns = [
        f"embedding_{column:03d}" for column in range(frame.shape[1])
    ]
    return frame


# 1. Load embeddings
embeddings = load_participant_embeddings(EMBEDDING_PATH)
embedding_columns = embeddings.columns.tolist()

# 2. Extract base ID for joining while preserving the original eid
embeddings = embeddings.reset_index()
embeddings["base_eid"] = embeddings["eid"].str.split("_").str[0]

# 3. Prepare demographic data
demo_cols = ["Local Participant", "age", "gender", "BMI", "healthstat", "smoker", "alcocons", "T25FW"]
tabular_subset = tabular[demo_cols].copy()
tabular_subset["Local Participant"] = tabular_subset[
    "Local Participant"
].astype(str)

# 4. Perform Many-to-One merge and restore 'eid' (12345_T1 format) as the index
cohort = embeddings.merge(
    tabular_subset,
    left_on="base_eid",
    right_on="Local Participant",
    how="inner",
).drop(columns=["base_eid", "Local Participant"])

# Re-index by full 'eid' to match downstream expectations
cohort = cohort.set_index("eid")

print(f"Participants with tabular data and embeddings: {len(cohort):,}")
cohort[["age", "gender", "BMI", "healthstat", "smoker", "alcocons", "T25FW"]].head()

Participants with tabular data and embeddings: 4,620


,age,gender,BMI,healthstat,smoker,alcocons,T25FW
eid,,,,,,,
10376_T1,50,1,22.790358,Good,1,2.0,11.6
10376_T1,50,1,22.790358,Good,1,2.0,12.9
10376_T1,50,1,22.790358,Good,1,2.0,11.4
10376_T1,50,1,22.790358,Good,1,2.0,8.3
10376_T1,50,1,22.790358,Good,1,2.0,12.0


## matching method from sensori paper (class weighting, folds etc)
- gemini gave me some code to evaluate all metrics

In [30]:

# Define seed for reproducibility
SEED = 71

# 1. Identify embedding feature columns automatically
feature_cols = cohort.filter(regex=r"^embedding_").columns

In [19]:


# # Define seed for reproducibility
# SEED = 71
#
# # 1. Identify embedding feature columns automatically
# feature_cols = cohort.filter(regex=r"^embedding_").columns


# =========================================================================
# A. CONTINUOUS TARGETS: L2-Regularized Linear Regression (Ridge)
# =========================================================================
def evaluate_continuous_target(df, target_col, feature_cols, n_splits=5):
    """Fits L2-regularized Ridge regression using GroupKFold cross-validation."""
    clean_df = df.dropna(subset=[target_col] + list(feature_cols)).copy()
    X = clean_df[feature_cols]
    y = clean_df[target_col]
    groups = clean_df.index.astype(str).str.split("_").str[0]

    gkf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(clean_df))

    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        # Standardize features strictly within each training fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # L2-regularized linear regression
        model = Ridge(alpha=100.0, random_state=SEED)
        model.fit(X_train_scaled, y_train)

        oof_preds[test_idx] = model.predict(X_test_scaled)

    # Calculate overall dataset-level evaluation metrics
    r_stat, p_val = pearsonr(y, oof_preds)
    r2 = r2_score(y, oof_preds)
    mae = mean_absolute_error(y, oof_preds)
    rmse = root_mean_squared_error(y, oof_preds)

    print(f"=== Continuous Evaluation: {target_col} ===")
    print(f"Pearson's r : {r_stat:.3f} (p-value: {p_val:.2e})")
    print(f"R² Score    : {r2:.3f}")
    print(f"MAE         : {mae:.2f}")
    print(f"RMSE        : {rmse:.2f}\n")

    return oof_preds


# =========================================================================
# B. CATEGORICAL (BINARY & MULTI-CLASS) TARGETS: Class-Balanced L2 Logistic Regression
# =========================================================================
def evaluate_categorical_target(df, target_col, feature_cols, n_splits=5):
    """Fits class-balanced L2-regularized Logistic Regression (binary or multiclass)

    using StratifiedGroupKFold cross-validation.
    """
    clean_df = df.dropna(subset=[target_col] + list(feature_cols)).copy()
    X = clean_df[feature_cols]

    # Encode string categories into numeric labels
    le = LabelEncoder()
    y = le.fit_transform(clean_df[target_col].astype(str))
    classes = le.classes_
    num_classes = len(classes)

    # Extract base participant ID for group splitting
    groups = clean_df.index.astype(str).str.split("_").str[0]

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=SEED
    )
    oof_probs = np.zeros((len(clean_df), num_classes))

    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y[train_idx]

        # Standardize features strictly within each training fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # L2-regularized Logistic Regression with balanced class weighting
        clf = LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=SEED,
        )
        clf.fit(X_train_scaled, y_train)

        oof_probs[test_idx] = clf.predict_proba(X_test_scaled)

    # Compute AUROC based on target cardinality
    if num_classes == 2:
        # Binary target (e.g. gender)
        overall_auroc = roc_auc_score(y, oof_probs[:, 1])
        print(
            f"=== Categorical Evaluation (Binary): {target_col} [{', '.join(classes)}] ==="
        )
        print(f"AUROC : {overall_auroc:.4f}\n")
    else:
        # Multi-class target (e.g. healthstat)
        overall_auroc = roc_auc_score(
            y, oof_probs, multi_class="ovr", average="macro"
        )
        print(
            f"=== Categorical Evaluation (Multi-class): {target_col} [{', '.join(classes)}] ==="
        )
        print(f"Macro OVR AUROC : {overall_auroc:.4f}")

        # Per-class AUROC breakdown
        for i, cls in enumerate(classes):
            cls_binary_y = (y == i).astype(int)
            cls_auroc = roc_auc_score(cls_binary_y, oof_probs[:, i])
            print(f"  └─ Class '{cls}' AUROC: {cls_auroc:.4f}")
        print()

    return oof_probs


# -------------------------------------------------------------------------
# Run linear probe evaluation pipeline on cohort
# -------------------------------------------------------------------------
oof_age_preds = evaluate_continuous_target(cohort, "age", feature_cols)
oof_bmi_preds = evaluate_continuous_target(cohort, "BMI", feature_cols)
oof_gender_probs = evaluate_categorical_target(cohort, "gender", feature_cols)
oof_healthstat_probs = evaluate_categorical_target(
    cohort, "healthstat", feature_cols
)
oof_smoking_probs = evaluate_categorical_target(
    cohort, "smoker", feature_cols
)
oof_alcohol_preds = evaluate_continuous_target(cohort, "alcocons", feature_cols)
oof_t25fw_preds = evaluate_continuous_target(cohort, "T25FW", feature_cols)


=== Continuous Evaluation: age ===
Pearson's r : 0.670 (p-value: 0.00e+00)
R² Score    : 0.421
MAE         : 6.61
RMSE        : 8.15

=== Continuous Evaluation: BMI ===
Pearson's r : 0.560 (p-value: 0.00e+00)
R² Score    : 0.263
MAE         : 4.10
RMSE        : 5.44

=== Categorical Evaluation (Binary): gender [0, 1] ===
AUROC : 0.8912

=== Categorical Evaluation (Multi-class): healthstat [Average, Bad, Good, Less than average, Very bad, Very good] ===
Macro OVR AUROC : 0.6305
  └─ Class 'Average' AUROC: 0.4769
  └─ Class 'Bad' AUROC: 0.5770
  └─ Class 'Good' AUROC: 0.5915
  └─ Class 'Less than average' AUROC: 0.4724
  └─ Class 'Very bad' AUROC: 0.8535
  └─ Class 'Very good' AUROC: 0.8114

=== Categorical Evaluation (Binary): smoker [0, 1] ===
AUROC : 0.6006

=== Continuous Evaluation: alcocons ===
Pearson's r : 0.021 (p-value: 1.54e-01)
R² Score    : -0.402
MAE         : 5.83
RMSE        : 7.79

=== Continuous Evaluation: T25FW ===
Pearson's r : 0.583 (p-value: 0.00e+00)
R² Score    :

In [34]:
demo_feats = ["age", "BMI", "smoker", "gender"]

In [20]:
# TODO: convert demographic features into format that is accepted by model
# demo_feats = ["age", "BMI", "smoker", "gender"]

oof_t25fw_preds = evaluate_continuous_target(cohort, "T25FW", demo_feats)

=== Continuous Evaluation: T25FW ===
Pearson's r : 0.100 (p-value: 1.70e-11)
R² Score    : 0.005
MAE         : 4.38
RMSE        : 7.53



In [35]:
all_features = demo_feats + feature_cols.tolist()
print(all_features)

['age', 'BMI', 'smoker', 'gender', 'embedding_000', 'embedding_001', 'embedding_002', 'embedding_003', 'embedding_004', 'embedding_005', 'embedding_006', 'embedding_007', 'embedding_008', 'embedding_009', 'embedding_010', 'embedding_011', 'embedding_012', 'embedding_013', 'embedding_014', 'embedding_015', 'embedding_016', 'embedding_017', 'embedding_018', 'embedding_019', 'embedding_020', 'embedding_021', 'embedding_022', 'embedding_023', 'embedding_024', 'embedding_025', 'embedding_026', 'embedding_027', 'embedding_028', 'embedding_029', 'embedding_030', 'embedding_031', 'embedding_032', 'embedding_033', 'embedding_034', 'embedding_035', 'embedding_036', 'embedding_037', 'embedding_038', 'embedding_039', 'embedding_040', 'embedding_041', 'embedding_042', 'embedding_043', 'embedding_044', 'embedding_045', 'embedding_046', 'embedding_047', 'embedding_048', 'embedding_049', 'embedding_050', 'embedding_051', 'embedding_052', 'embedding_053', 'embedding_054', 'embedding_055', 'embedding_05

In [36]:
oof_t25fw_preds = evaluate_continuous_target(cohort, "T25FW", all_features)


=== Continuous Evaluation: T25FW ===
Pearson's r : 0.581 (p-value: 0.00e+00)
R² Score    : 0.298
MAE         : 4.22
RMSE        : 6.33

